In [7]:

# -*- coding: utf-8 -*-
"""
Bước 1: Tự tính khoảng cách và tự tìm láng giềng, chưa dùng thư viện.
Chú thích viết tiếng Việt có dấu, phần in ra màn hình viết không dấu.
"""

import numpy as np
import pandas as pd

# Ba bạn: A và C học gần như y hệt nhau, B học khác hẳn
ba_ban = {
    "A": np.array([5.0, 3.0, 4000.0]),
    "B": np.array([25.0, 9.0, 4100.0]),
    "C": np.array([6.0, 3.5, 7000.0]),
}


def kc(u, v):
    """Khoảng cách Euclid giữa hai điểm."""
    return np.sqrt(((u - v) ** 2).sum())


print("Khoang cach KHI CHUA chuan hoa")
print(f" d(A, B) = {kc(ba_ban['A'], ba_ban['B']):8.1f}")
print(f" d(A, C) = {kc(ba_ban['A'], ba_ban['C']):8.1f}")
print()

# Chuẩn hóa bằng trung bình và độ lệch chuẩn của cả bộ dữ liệu
df = pd.read_csv("data/sinh_vien_mo_rong.csv")
X = df[["gio_on", "diem_giua_ky", "chi_tieu"]].to_numpy(dtype=float)

tb = X.mean(axis=0)
dl = X.std(axis=0)

print("Do lech chuan cua ba cot:")
print(
    f" gio_on {dl[0]:.2f}    diem_giua_ky {dl[1]:.2f}"
    f"   chi_tieu {dl[2]:.0f}"
)
print()

chuan = {ten: (v - tb) / dl for ten, v in ba_ban.items()}

print("Khoang cach SAU KHI chuan hoa")
print(f" d(A, B) = {kc(chuan['A'], chuan['B']):.2f}")
print(f" d(A, C) = {kc(chuan['A'], chuan['C']):.2f}")
print()

# Tìm 5 láng giềng gần nhất của một bạn mới, chỉ dùng hai cột học tập
X2 = df[["gio_on", "diem_giua_ky"]].to_numpy(dtype=float)
y = df["qua_mon"].to_numpy()

tb2, dl2 = X2.mean(axis=0), X2.std(axis=0)
ban_moi = np.array([12.0, 6.5])

X2c = (X2 - tb2) / dl2
moi_c = (ban_moi - tb2) / dl2

khoang = np.sqrt(((X2c - moi_c) ** 2).sum(axis=1))
gan = np.argsort(khoang)[:5]

print("Ban moi: on 12 gio, diem giua ky 6.5")
print("Nam lang gieng gan nhat:")

for i in gan:
    ket = "qua" if y[i] == 1 else "rot"
    print(
        f" gio {X2[i, 0]:5.1f} diem {X2[i, 1]:4.1f}"
        f"       khoang cach {khoang[i]:.3f} -> {ket}"
    )

so_qua = int(y[gan].sum())

print(f"Bo phieu: {so_qua} qua, {5 - so_qua} rot")
print("Du doan:", "qua mon" if so_qua >= 3 else "rot mon")


Khoang cach KHI CHUA chuan hoa
 d(A, B) =    102.2
 d(A, C) =   3000.0

Do lech chuan cua ba cot:
 gio_on 7.82    diem_giua_ky 2.09   chi_tieu 1692

Khoang cach SAU KHI chuan hoa
 d(A, B) = 3.85
 d(A, C) = 1.79

Ban moi: on 12 gio, diem giua ky 6.5
Nam lang gieng gan nhat:
 gio  10.5 diem  6.4       khoang cach 0.198 -> qua
 gio  12.5 diem  6.1       khoang cach 0.202 -> qua
 gio  14.0 diem  6.4       khoang cach 0.260 -> qua
 gio   9.5 diem  6.1       khoang cach 0.373 -> rot
 gio  10.5 diem  5.8       khoang cach 0.386 -> qua
Bo phieu: 4 qua, 1 rot
Du doan: qua mon


In [8]:

# -*- coding: utf-8 -*-
"""Bước 2: Đo hậu quả của việc quên chuẩn hóa trên tập kiểm tra."""

import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("data/sinh_vien_mo_rong.csv")
y = df["qua_mon"]

X3 = df[["gio_on", "diem_giua_ky", "chi_tieu"]]

X_tr, X_te, y_tr, y_te = train_test_split(
    X3,
    y,
    test_size=0.25,
    random_state=17,
    stratify=y
)


def cham(mo_hinh, cot):
    mo_hinh.fit(X_tr[cot], y_tr)
    return accuracy_score(y_te, mo_hinh.predict(X_te[cot]))


def knn_chuan():
    """KNN có chuẩn hóa, gộp làm một khối bằng pipeline."""
    return make_pipeline(
        StandardScaler(),
        KNeighborsClassifier(7)
    )


ba_cot = ["gio_on", "diem_giua_ky", "chi_tieu"]
hai_cot = ["gio_on", "diem_giua_ky"]

a = cham(KNeighborsClassifier(n_neighbors=7), ba_cot)
b = cham(knn_chuan(), ba_cot)
c = cham(knn_chuan(), hai_cot)

print(f"Doan bua ca lop qua              : {y_te.mean():.4f}")
print(f"Ba cot, khong chuan hoa          : {a:.4f}")
print(f"Ba cot, co chuan hoa             : {b:.4f}")
print(f"Bo cot chi_tieu, co chuan hoa    : {c:.4f}")


Doan bua ca lop qua              : 0.6000
Ba cot, khong chuan hoa          : 0.5333
Ba cot, co chuan hoa             : 0.8667
Bo cot chi_tieu, co chuan hoa    : 0.9000


In [9]:

# -*- coding: utf-8 -*-
"""Bước 3: Chọn số láng giềng k bằng kiểm định chéo."""

import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsClassifier

df = pd.read_csv("data/hai_vang_trang.csv")

X = df[["x1", "x2"]]
y = df["nhan"]

X_tr, X_te, y_tr, y_te = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=17,
    stratify=y
)

# Hai cột x1 và x2 có thang đo gần nhau,
# nên ở bộ dữ liệu này không chuẩn hóa cũng được.
print("Accuracy tren tap hoc va tap kiem tra theo k")
print("   k    tap hoc   tap kiem tra")

for k in [1, 5, 13, 31, 101]:
    mh = KNeighborsClassifier(n_neighbors=k).fit(X_tr, y_tr)

    a_hoc = accuracy_score(y_tr, mh.predict(X_tr))
    a_kt = accuracy_score(y_te, mh.predict(X_te))

    print(f" {k:3d}    {a_hoc:.4f}      {a_kt:.4f}")

print()

# Chọn k chỉ dựa trên tập học, không nhìn tập kiểm tra
print("Kiem dinh cheo 5 phan tren tap hoc:")

diem = {}

for k in range(1, 52, 2):
    diem[k] = cross_val_score(
        KNeighborsClassifier(n_neighbors=k),
        X_tr,
        y_tr,
        cv=5
    ).mean()

k_tot = max(diem, key=diem.get)

for k in [1, 5, 9, 13, 17, 25, 51]:
    dau = " <- tot nhat" if k == k_tot else ""
    print(f" k = {k:2d} diem trung binh {diem[k]:.4f}{dau}")

print()

# Chọn xong mới đánh giá trên tập kiểm tra
mh = KNeighborsClassifier(n_neighbors=k_tot).fit(X_tr, y_tr)

print(f"k tot nhat = {k_tot}")
acc = accuracy_score(y_te, mh.predict(X_te))
print(f"Accuracy tren tap kiem tra = {acc:.4f}")


Accuracy tren tap hoc va tap kiem tra theo k
   k    tap hoc   tap kiem tra
   1    1.0000      0.8000
   5    0.8756      0.9067
  13    0.8844      0.9200
  31    0.8667      0.8933
 101    0.8311      0.8533

Kiem dinh cheo 5 phan tren tap hoc:
 k =  1 diem trung binh 0.7778
 k =  5 diem trung binh 0.8444
 k =  9 diem trung binh 0.8444
 k = 13 diem trung binh 0.8667 <- tot nhat
 k = 17 diem trung binh 0.8533
 k = 25 diem trung binh 0.8622
 k = 51 diem trung binh 0.8533

k tot nhat = 13
Accuracy tren tap kiem tra = 0.9200


In [10]:

# -*- coding: utf-8 -*-
"""Bước 4: So sánh KNN với hai mô hình đã học trên dữ liệu có biên cong."""

import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier

df = pd.read_csv("data/hai_vang_trang.csv")

X = df[["x1", "x2"]]
y = df["nhan"]

X_tr, X_te, y_tr, y_te = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=17,
    stratify=y
)

cac_mo_hinh = [
    ("Hoi quy logistic", LogisticRegression()),
    ("Naive Bayes     ", GaussianNB()),
    ("KNN, k = 13     ", KNeighborsClassifier(n_neighbors=13)),
]

print("Accuracy tren tap kiem tra, bo hai vang trang")

for ten, mh in cac_mo_hinh:
    mh.fit(X_tr, y_tr)
    du_doan = mh.predict(X_te)

    print(f" {ten}: {accuracy_score(y_te, du_doan):.4f}")


Accuracy tren tap kiem tra, bo hai vang trang
 Hoi quy logistic: 0.8533
 Naive Bayes     : 0.8533
 KNN, k = 13     : 0.9200


In [11]:

# -*- coding: utf-8 -*-
"""Bước 5: KNN cho bài toán hồi quy trên dữ liệu giá nhà."""

import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsRegressor

df = pd.read_csv("data/gia_nha.csv")

X = df[["dien_tich"]]
y = df["gia"]

# Cùng cách chia với bài 1 để so sánh công bằng
X_tr, X_te, y_tr, y_te = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

knn = KNeighborsRegressor(n_neighbors=5).fit(X_tr, y_tr)
duong = LinearRegression().fit(X_tr, y_tr)

print("MAE tren tap kiem tra (ty dong)")

mae_knn = mean_absolute_error(y_te, knn.predict(X_te))
mae_duong = mean_absolute_error(y_te, duong.predict(X_te))

print(f" KNN, k = 5 : {mae_knn:.4f}")
print(f" Duong thang : {mae_duong:.4f}")
print()

# Dự đoán giá cho bốn căn nhà mới
can_moi = pd.DataFrame({
    "dien_tich": [60.0, 100.0, 150.0, 250.0]
})

print("Du doan cho bon can moi (ty dong)")
print(" dien tich      KNN   duong thang")

for dt, a, b in zip(
    can_moi["dien_tich"],
    knn.predict(can_moi),
    duong.predict(can_moi)
):
    print(f" {dt:6.0f} m2    {a:6.2f}   {b:6.2f}")

print()

# Năm láng giềng của căn 250 m2
_, chi_so = knn.kneighbors(
    pd.DataFrame({"dien_tich": [250.0]})
)

lg = X_tr.iloc[chi_so[0]]["dien_tich"].tolist()

print(
    "Nam lang gieng cua can 250 m2:",
    ", ".join(f"{v:.1f}" for v in lg)
)


MAE tren tap kiem tra (ty dong)
 KNN, k = 5 : 0.4030
 Duong thang : 0.2578

Du doan cho bon can moi (ty dong)
 dien tich      KNN   duong thang
     60 m2      4.95     5.12
    100 m2      8.32     8.24
    150 m2      9.32    12.13
    250 m2      9.32    19.91

Nam lang gieng cua can 250 m2: 117.5, 115.9, 111.7, 111.2, 104.2


Bài tập 1

In [12]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

def knn_du_doan(X_hoc, y_hoc, x_moi, k):
    khoang_cach = np.sqrt(((X_hoc - x_moi) ** 2).sum(axis=1))
    gan_nhat = np.argsort(khoang_cach)[:k]
    nhan, dem = np.unique(np.asarray(y_hoc)[gan_nhat], return_counts=True)
    return nhan[np.argmax(dem)]

df = pd.read_csv("data/hai_vang_trang.csv")
X = df[["x1", "x2"]].to_numpy(dtype=float)
y = df["nhan"].to_numpy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

y_pred = np.array([
    knn_du_doan(X_train, y_train, x, 13) for x in X_test
])
print("Accuracy KNN tự viết:", accuracy_score(y_test, y_pred))

model = KNeighborsClassifier(n_neighbors=13)
model.fit(X_train, y_train)
print("Accuracy thư viện:", accuracy_score(y_test, model.predict(X_test)))

Accuracy KNN tự viết: 0.92
Accuracy thư viện: 0.92


- Tự tính khoảng cách Euclid, chọn 13 láng giềng gần nhất và bỏ phiếu để dự đoán nhãn.
- Accuracy của KNN tự viết cần bằng accuracy của KNeighborsClassifier khi cách xử lý trường hợp hòa phiếu giống nhau.

Bài tập 2

In [13]:
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

df = pd.read_csv("data/hai_vang_trang.csv")
X, y = df[["x1", "x2"]], df["nhan"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

for metric in ["euclidean", "manhattan"]:
    scores = []
    for k in range(1, 31):
        model = KNeighborsClassifier(n_neighbors=k, metric=metric)
        scores.append(cross_val_score(model, X_train, y_train, cv=5).mean())

    best_k = scores.index(max(scores)) + 1
    model = KNeighborsClassifier(n_neighbors=best_k, metric=metric)
    model.fit(X_train, y_train)
    acc = accuracy_score(y_test, model.predict(X_test))

    print(f"{metric}: k tốt nhất = {best_k}, accuracy = {acc:.4f}")

euclidean: k tốt nhất = 26, accuracy = 0.9200
manhattan: k tốt nhất = 27, accuracy = 0.9067


- Khoảng cách Euclid đo khoảng cách đường thẳng, còn Manhattan tính tổng độ chênh lệch tuyệt đối.
- So sánh accuracy in ra để xác định phương pháp nào phù hợp hơn với dữ liệu.

Bài tập 3

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("data/sinh_vien_mo_rong.csv")
X = df[["gio_on", "diem_giua_ky", "chi_tieu"]].copy()
y = df["qua_mon"]

# Đổi chi tiêu từ nghìn đồng sang triệu đồng
X["chi_tieu"] = X["chi_tieu"] / 1000

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

# Không chuẩn hóa
model1 = KNeighborsClassifier(n_neighbors=7)
model1.fit(X_train, y_train)
acc1 = accuracy_score(y_test, model1.predict(X_test))

# Có chuẩn hóa
model2 = make_pipeline(
    StandardScaler(), KNeighborsClassifier(n_neighbors=7)
)
model2.fit(X_train, y_train)
acc2 = accuracy_score(y_test, model2.predict(X_test))

print(f"Không chuẩn hóa: {acc1:.4f}")
print(f"Có chuẩn hóa: {acc2:.4f}")

- Khi không chuẩn hóa, đổi đơn vị cột chi tiêu có thể làm thay đổi đáng kể khoảng cách và accuracy.
- Khi chuẩn hóa đúng cách, đổi đơn vị đo thường không làm thay đổi kết quả KNN, ngoại trừ sai số tính toán nhỏ.

Bài tập 4

In [14]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

df = pd.read_csv("data/hai_vang_trang.csv")
X, y = df[["x1", "x2"]], df["nhan"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

for k in [1, 13, 101]:
    for weights in ["uniform", "distance"]:
        model = KNeighborsClassifier(n_neighbors=k, weights=weights)
        model.fit(X_train, y_train)
        acc = accuracy_score(y_test, model.predict(X_test))
        print(f"k={k}, weights={weights}, accuracy={acc:.4f}")

k=1, weights=uniform, accuracy=0.8000
k=1, weights=distance, accuracy=0.8000
k=13, weights=uniform, accuracy=0.9200
k=13, weights=distance, accuracy=0.9333
k=101, weights=uniform, accuracy=0.8533
k=101, weights=distance, accuracy=0.9067


- uniform cho mỗi láng giềng một phiếu như nhau; distance ưu tiên các láng giềng gần hơn.
- Với \(k=101\), trọng số theo khoảng cách có thể giúp cải thiện dự đoán, nhưng cần dựa vào accuracy thực tế để kết luận.

Bài tập 5

In [15]:
import pandas as pd
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

df = pd.read_csv("data/gia_nha.csv")
X, y = df[["dien_tich"]], df["gia"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

cv = KFold(n_splits=5, shuffle=True, random_state=42)
ket_qua = []

for k in range(1, 21):
    model = KNeighborsRegressor(n_neighbors=k)
    scores = cross_val_score(
        model, X_train, y_train,
        cv=cv, scoring="neg_mean_absolute_error"
    )
    ket_qua.append((-scores.mean(), k))

mae_cv, best_k = min(ket_qua)

knn = KNeighborsRegressor(n_neighbors=best_k)
knn.fit(X_train, y_train)

linear = LinearRegression()
linear.fit(X_train, y_train)

mae_knn = mean_absolute_error(y_test, knn.predict(X_test))
mae_linear = mean_absolute_error(y_test, linear.predict(X_test))

print("k tốt nhất:", best_k)
print(f"MAE kiểm định chéo: {mae_cv:.4f}")
print(f"MAE KNN: {mae_knn:.4f}")
print(f"MAE hồi quy tuyến tính: {mae_linear:.4f}")

k tốt nhất: 2
MAE kiểm định chéo: 0.3587
MAE KNN: 0.5200
MAE hồi quy tuyến tính: 0.2917


- Sử dụng kiểm định chéo 5 phần để chọn \(k\) có MAE nhỏ nhất trong khoảng từ 1 đến 20.
- MAE càng thấp thì dự đoán càng tốt. So sánh MAE của KNN với hồi quy tuyến tính để xác định mô hình phù hợp hơn.

Bài tập 6

In [16]:
import time
import numpy as np
from sklearn.neighbors import KNeighborsClassifier

rng = np.random.default_rng(0)
X_moi = rng.random((1000, 2))
y_moi = rng.integers(0, 2, 1000)

print(f"{'Số mẫu':>10} {'Fit (giây)':>15} {'Predict (giây)':>18}")

for n in [1000, 10000, 100000]:
    X = rng.random((n, 2))
    y = rng.integers(0, 2, n)

    model = KNeighborsClassifier(n_neighbors=5)

    start = time.perf_counter()
    model.fit(X, y)
    fit_time = time.perf_counter() - start

    start = time.perf_counter()
    model.predict(X_moi)
    predict_time = time.perf_counter() - start

    print(f"{n:10d} {fit_time:15.6f} {predict_time:18.6f}")

    Số mẫu      Fit (giây)     Predict (giây)
      1000        0.001782           0.006076
     10000        0.004865           0.004314
    100000        0.061488           0.006575


- Thời gian huấn luyện KNN thường ngắn vì mô hình chủ yếu lưu dữ liệu.
- Khi số mẫu tăng, thời gian dự đoán thường tăng do phải tìm láng giềng gần nhất.
- Vì phần lớn tính toán diễn ra lúc dự đoán thay vì lúc huấn luyện, KNN được gọi là mô hình học lười (Lazy Learning).